# 03 · Model evaluation

CNN vs LightGBM on held-out stars: metrics, confusion matrices, PR curves, calibration, the hardest mistakes, and what the CNN looked at.

Run `make train` first.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

from transit_hunter.config import load_data_config, setup_logging

setup_logging("WARNING")
cfg = load_data_config()
plt.rcParams.update({"figure.figsize": (12, 3.5), "axes.grid": True, "grid.alpha": 0.3})
BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#199e70"

In [ ]:
import json
import pandas as pd
from transit_hunter.config import CLASSES, CLASS_NAMES, get_settings, REPO_ROOT
from transit_hunter.data.dataset import load_dataset
from transit_hunter.models.predict import load_predictor, model_paths
from transit_hunter.models.evaluate import evaluate_predictions, comparison_table

ds = load_dataset()
paths = model_paths()
print("models:", {k: str(v) for k, v in paths.items()})
SPLIT = "test"
sub = ds.subset(SPLIT, real_only=True)
if len(sub) == 0:
    SPLIT, sub = "val", ds.subset("val")
print(f"evaluating on {SPLIT}: {len(sub)} examples from {sub.index['star_id'].nunique()} stars")

preds, metrics = {}, {}
for kind in paths:
    pred = load_predictor(kind)
    preds[kind] = pred.predict_proba(sub.global_views, sub.local_views, sub.features())
    metrics[kind] = {SPLIT: evaluate_predictions(sub.labels, preds[kind])}
print(comparison_table(metrics, SPLIT))

## Confusion matrices

In [ ]:
fig, axes = plt.subplots(1, len(preds), figsize=(5 * len(preds), 4))
axes = np.atleast_1d(axes)
for ax, (kind, m) in zip(axes, metrics.items()):
    cm = np.array(m[SPLIT]["confusion_matrix"])
    ax.imshow(cm, cmap="Blues")
    for i in range(3):
        for j in range(3):
            ax.text(j, i, cm[i, j], ha="center", va="center", color="white" if cm[i, j] > cm.max() / 2 else "black")
    ax.set_xticks(range(3), CLASSES); ax.set_yticks(range(3), CLASSES)
    ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title(kind.upper()); ax.grid(False)
plt.tight_layout(); plt.show()

## Precision-recall per class, and calibration of P(planet)

In [ ]:
from sklearn.metrics import precision_recall_curve

fig, axes = plt.subplots(1, 4, figsize=(16, 3.8))
styles = {"cnn": "-", "gbm": "--"}
for k, cls in enumerate(CLASSES):
    y = (sub.labels == k).astype(int)
    for kind, p in preds.items():
        if y.sum():
            pr, rc, _ = precision_recall_curve(y, p[:, k])
            axes[k].plot(rc, pr, styles.get(kind, "-"), label=kind)
    axes[k].set_title(CLASS_NAMES[cls]); axes[k].set_xlabel("recall"); axes[k].set_ylabel("precision")
axes[0].axhline(0.95, color="0.5", ls=":", lw=1)
axes[0].legend()
for kind, m in metrics.items():
    cal = m[SPLIT]["calibration_pc"]
    xs = [a for a in cal["mean_predicted"] if a is not None]; ys = [b for b in cal["fraction_positive"] if b is not None]
    axes[3].plot(xs, ys, "o" + styles.get(kind, "-"), label=f"{kind} (ECE {cal['ece']:.3f})")
axes[3].plot([0, 1], [0, 1], color="0.6", ls=":"); axes[3].set_title("Calibration of P(planet)")
axes[3].set_xlabel("predicted"); axes[3].set_ylabel("observed fraction"); axes[3].legend(fontsize=8)
plt.tight_layout(); plt.show()

## The most confident mistakes

Often label noise (see the data card), sometimes a genuine model blind spot. Worth reading before the judges ask.

In [ ]:
kind = "cnn" if "cnn" in preds else next(iter(preds))
p = preds[kind]
pred_cls = p.argmax(1)
wrong = np.flatnonzero(pred_cls != sub.labels)
worst = wrong[np.argsort(-p[wrong].max(1))][:6]
cols = ["sample_id", "star_id", "label", "label_reason", "period", "duration", "measured_depth_ppm"]
table = sub.index.iloc[worst][[c for c in cols if c in sub.index]].copy()
table["predicted"] = [CLASSES[i] for i in pred_cls[worst]]
table["confidence"] = p[worst].max(1).round(3)
display(table)

In [ ]:
from transit_hunter.models.explain import explain

if "cnn" in paths and len(worst):
    cnn = load_predictor("cnn")
    fig, axes = plt.subplots(len(worst), 2, figsize=(12, 2.2 * len(worst)), squeeze=False)
    for r, i in enumerate(worst):
        e = explain(cnn, sub.global_views[i], sub.local_views[i], sub.features()[i])
        for ax, view, mag, name in ((axes[r, 0], sub.global_views[i], e.global_magnitude, "global"),
                                    (axes[r, 1], sub.local_views[i], e.local_magnitude, "local")):
            ax.plot(view, color="0.6", lw=1)
            ax.scatter(range(len(view)), view, c=mag, cmap="Blues", s=8 + 30 * np.asarray(mag), vmin=0, vmax=1)
            ax.set_title(f"{sub.index['sample_id'].iloc[i]} true={sub.index['label'].iloc[i]} pred={e.target_class} ({name})", fontsize=9)
    plt.tight_layout(); plt.show()

## TESS domain shift (if `make data-tess` was run)

In [ ]:
tess = ds.subset("tess")
if len(tess):
    for kind in paths:
        m = evaluate_predictions(tess.labels, load_predictor(kind).predict_proba(tess.global_views, tess.local_views, tess.features()))
        print(f"{kind}: TESS macro PR-AUC {m['macro_pr_auc']:.3f}, PC recall {m['per_class']['PC']['recall']:.3f} (n={m['n']})")
else:
    print("No TESS split; run `make data-tess` to measure the Kepler -> TESS drop.")

## GBM feature importance

In [ ]:
fi_path = get_settings().models_dir / "gbm" / "feature_importance.json"
fi_path = fi_path if fi_path.is_absolute() else REPO_ROOT / fi_path
if fi_path.exists():
    fi = pd.Series(json.loads(fi_path.read_text())).sort_values().tail(15)
    fi.plot.barh(color=BLUE, figsize=(8, 5)); plt.title("LightGBM gain importance"); plt.tight_layout(); plt.show()